# ChromaVision
## Segmentation couleur, robustesse et frugalité

**Objectif :** comprendre comment des méthodes classiques segmentent des régions
cutanées annotées, puis comparer leurs erreurs et leurs coûts.

Ce notebook accompagne un projet complet : code, données du TP, résultats,
rapport et présentation. Ouvrir le notebook depuis le dossier décompressé.

**Équipe :** Pote Die et Hélène Bobe Douchet.
**Contributions :** contribution équivalente à la préparation des données,
à l'implémentation, aux expériences, à l'analyse et à la rédaction.
Le lien du dépôt réel reste à renseigner avant la remise.

## Version publique
Ce notebook décrit la méthode complète. Les photographies et leurs annotations
ne sont pas distribuées dans le dépôt. Pour le lancer intégralement, importer
les données du TP avec `scripts/import_lab.py`. Pour consulter les résultats
sans télécharger de photographies, ouvrir **00_Resultats_Publics.ipynb**.
Les figures qualitatives sont produites uniquement dans la copie locale.
Après import, lancer aussi `python scripts/make_figures.py` depuis la racine pour générer les illustrations locales avant de parcourir ce notebook.

### 1. Préparer les imports
Installer les dépendances avec `python -m pip install -r requirements.txt`
dans le terminal du projet, puis sélectionner le même Python dans VS Code.
Toutes les images passent en **RGB**. Les masques sont des tableaux booléens.

In [ ]:
from pathlib import Path
import sys, json, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = Path.cwd().resolve()
if not (ROOT / "src/chromavision").exists():
    ROOT = ROOT.parent
assert (ROOT / "src/chromavision").exists(), "Ouvrir le notebook dans ChromaVision/notebooks"
sys.path.insert(0, str(ROOT / "src"))
from chromavision.core import *
cv2.setNumThreads(1)
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 10})
print("Projet trouvé. OpenCV :", cv2.__version__)

### 2. Vérifier les données
Le TP contient 71 paires valides. Ce n'est pas l'ensemble officiel de 78 images
parfois décrit dans la littérature. L'importeur a corrigé une association de
noms et garde les empreintes des fichiers. Aucune image de test ne sert à
choisir les paramètres. Les identités ne sont pas annotées : une séparation
par image ne prouve pas une séparation par personne.

In [ ]:
manifest = pd.read_csv(ROOT / "results/manifest_split.csv")
display(manifest.groupby(["split", "category"]).size().unstack())
assert manifest.id.is_unique
assert manifest.sha256_pixels.is_unique
audit = json.loads((ROOT / "results/audit.json").read_text())
display(audit)

In [ ]:
# Une photographie de validation et son annotation, sans l'utiliser pour apprendre.
record = manifest[manifest["split"] == "validation"].iloc[0].to_dict()
rgb, truth = read_pair(ROOT, record)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(rgb); axes[0].set_title("Photographie RGB")
axes[1].imshow(truth, cmap="gray"); axes[1].set_title("Annotation : blanc = positif")
for ax in axes: ax.axis("off")
plt.show()
print("Dimensions d'analyse :", rgb.shape, "Proportion positive :", round(truth.mean(), 3))

### 3. Question et hypothèses
**Question :** séparer luminosité et chrominance, nettoyer les masques ou
rééquilibrer les couleurs améliore-t-il le compromis entre IoU et coût ?

- H1 : les histogrammes chromatiques résistent mieux à un assombrissement que la règle RGB.
- H2 : la morphologie réduit les erreurs isolées, au prix possible de petites régions perdues.
- H3 : Gray World peut atténuer une dominante chaude mais peut dégrader d'autres scènes.
- Frugalité : les modèles par tables de couleurs sont petits ; le processus Python complet reste plus volumineux.

Ce sont des hypothèses à discuter à partir des observations, pas des conclusions annoncées à l'avance.

### 4. Comprendre la référence RGB
Un pixel est positif si ses composantes satisfont plusieurs inégalités, par
exemple `R > 95`, `R > G` et `R > B`. Cette règle n'apprend rien à partir des
photographies. Une surface du décor de couleur voisine peut donc la tromper.

In [ ]:
toy = np.array([[[180, 100, 50], [10, 200, 50], [20, 30, 200]]], dtype=np.uint8)
print("Trois pixels RGB :", toy.tolist())
print("Prédiction :", rgb_rule(toy).tolist())

### 5. Comprendre les histogrammes
On compte les couleurs dans deux classes : peau et fond. Les fréquences
normalisées donnent un score sous a priori équilibré :

$$s(c)=\frac{p(c\mid\mathrm{peau})}{p(c\mid\mathrm{peau})+p(c\mid\mathrm{fond})}.$$

La décision est $s(c)\geq\tau$. On choisit $\tau$ sur la validation et jamais
sur le test. La table comporte $B\times B$ cases, avec $B=16$ ou $32$.
Avec 32 cases par axe et un float32 par case, la LUT finale vaut 4 096 octets.
Ce score n'est pas une probabilité calibrée à la proportion réelle de peau.

In [ ]:
model = ColorModel.load(ROOT / "results/models", "Hist-Lab")
display({k: v for k, v in model.__dict__.items() if k != "lut"})
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(model.lut.T, origin="lower", vmin=0, vmax=1, cmap="viridis")
ax.set(xlabel="Case de l'axe a*", ylabel="Case de l'axe b*", title="Table de scores Lab apprise")
fig.colorbar(im, ax=ax); plt.show()
print("Taille de la table :", model.nbytes, "octets")

In [ ]:
# Refaire réellement l'apprentissage Lab dans le notebook, sur les 42 images train.
train_records = manifest[manifest["split"] == "train"].to_dict("records")
training_pairs = [(r["id"], *read_pair(ROOT, r)) for r in train_records]
fitted = fit_histogram(training_pairs, "Demo-Lab", space="Lab",
                       bins=model.bins, smooth=model.smooth)
np.testing.assert_allclose(fitted.lut, model.lut, atol=1e-7)
print("Histogramme réappris sur", len(training_pairs), "images. LUT identique au modèle livré.")

### 6. Choisir les réglages sans fuite de données
Quatre variantes d'histogrammes explorent 2 discrétisations, 2 lissages et
19 seuils. La morphologie explore ensuite 3 tailles de noyau. Le critère est
l'IoU moyenne sur les 14 photographies de validation originales.
Le fichier `validation_grid.csv` conserve tous les candidats, pas uniquement le gagnant.

In [ ]:
grid = pd.read_csv(ROOT / "results/validation_grid.csv")
display(grid.groupby("method").size().rename("candidats"))
display(grid.loc[grid.groupby("method").val_iou.idxmax()])
display(Image(filename=str(ROOT / "results/figures/validation_thresholds.png")))

In [ ]:
# Choisir effectivement le seuil de ce modèle Lab, sans consulter le test.
validation_records = manifest[manifest["split"] == "validation"].to_dict("records")
validation_scores = []
for r in validation_records:
    x, y = read_pair(ROOT, r)
    validation_scores.append((y, fitted.score(x)))
trials = []
for threshold in np.linspace(0.05, 0.95, 19):
    mean_iou = np.mean([binary_metrics(y, score >= threshold)["iou"]
                        for y, score in validation_scores])
    trials.append({"threshold": threshold, "val_iou": mean_iou})
trials = pd.DataFrame(trials)
best = trials.loc[trials.val_iou.idxmax()]
fitted.threshold = float(best.threshold)
assert abs(fitted.threshold - model.threshold) < 1e-12
display(best.to_frame("Sélection sur validation"))

### 7. Appliquer les méthodes et mesurer une erreur
`TP` = pixels positifs retrouvés, `FP` = pixels du fond ajoutés, `FN` = pixels
positifs manqués. **IoU = TP/(TP+FP+FN)**. Le Dice vaut **2TP/(2TP+FP+FN)**.
On calcule d'abord le score pour chaque photo, puis la moyenne : chaque photo
compte autant, quelle que soit sa taille.

In [ ]:
# Exemple vérifiable à la main : 80 vrais positifs, 20 faux positifs, 20 manqués.
t = np.array([1]*100 + [0]*100, dtype=bool)
p = np.array([1]*80 + [0]*20 + [1]*20 + [0]*80, dtype=bool)
display(binary_metrics(t, p))
assert abs(binary_metrics(t, p)["iou"] - 2/3) < 1e-12

In [ ]:
results = pd.read_csv(ROOT / "results/predictions.csv")
summary = pd.read_csv(ROOT / "results/summary.csv")
display(summary[summary.condition == "original"].sort_values("iou", ascending=False).round(3))
display(Image(filename=str(ROOT / "results/figures/clean_iou.png")))

### 8. Interpréter les intervalles de confiance
Les barres utilisent 3 000 rééchantillonnages des 15 **images** de test, à
modèles fixes. Les pixels d'une même photo ne sont pas indépendants.
Ces IC ne couvrent pas le changement de jeu de données ni toutes les séparations
possibles. Comparer les méthodes avec les contrastes appariés, pas seulement
avec le chevauchement de leurs barres. Aucun test confirmatoire corrigé pour
comparaisons multiples n'est revendiqué.

In [ ]:
contrasts = pd.read_csv(ROOT / "results/paired_contrasts.csv")
display(contrasts[contrasts.condition == "original"].round(3))

### 9. Tester la robustesse
Même photo et même annotation, mais image plus sombre, plus claire, plus chaude,
bruitée ou compressée. Les transformations s'appliquent après redimensionnement.
Les gamma s'appliquent aux valeurs RGB encodées, ce qui ne reproduit pas toute
la physique d'un changement réel d'éclairage.

In [ ]:
display(Image(filename=str(ROOT / "results/figures/perturbations.png")))
display(Image(filename=str(ROOT / "results/figures/robustness.png")))
display(contrasts[(contrasts.a.isin(["Lab-Morph", "GrayWorld-Lab"])) &
                  contrasts.condition.isin(["noise", "warm"])].round(3))

### 10. Regarder les échecs
La figure montre le pire cas, un cas médian et le meilleur cas de Hist-Lab
sur le test d'origine. Elle n'illustre donc pas seulement des réussites.
Chercher les objets du décor confondus avec la peau et les petites régions
perdues. Ne pas attribuer ces erreurs à un groupe démographique : le jeu ne
contient pas les annotations nécessaires pour cette analyse.

In [ ]:
display(Image(filename=str(ROOT / "results/figures/qualitative_cases.png")))

### 11. Mesurer la frugalité
La latence inclut les conversions et la morphologie, mais pas la lecture du
fichier. Chaque image reçoit un échauffement puis trois mesures, avec un thread
OpenCV. Les temps dépendent de l'ordinateur. La mémoire distingue la taille du
modèle et la RSS de tout le processus. La RSS est échantillonnée toutes les
1 ms et peut manquer un pic très bref. Ce n'est pas une mesure d'énergie.

In [ ]:
display(Image(filename=str(ROOT / "results/figures/quality_latency.png")))
memory = pd.read_csv(ROOT / "results/memory.csv")
display(memory[["method", "model_bytes", "sampled_peak_rss_mib", "sampled_delta_mib"]].round(3))

### 12. Réduire les annotations d'apprentissage
La courbe utilise 4, 12, 24 puis 42 images, avec trois sous-échantillonnages.
Chaque budget refait son apprentissage et son choix de bins, lissage et seuil,
avec les mêmes **14 images supplémentaires de validation**. Il ne réutilise
pas le modèle appris sur tout l'ensemble. Les courbes restent exploratoires :
trois tirages et 15 images de test ne suffisent pas à prouver un optimum de budget.

In [ ]:
eff = pd.read_csv(ROOT / "results/data_efficiency.csv")
display(eff.groupby(["budget", "seed"]).iou.mean().unstack().round(3))
display(Image(filename=str(ROOT / "results/figures/data_efficiency.png")))

### 13. Reproduire et vérifier
Laisser `REEXECUTER = False` pour lire les résultats fournis. Mettre `True`
pour relancer les calculs et remplacer les CSV locaux. La vérification finale
contrôle l'intégrité des données, les séparations et la cohérence des métriques.

In [ ]:
REEXECUTER = False
if REEXECUTER:
    for script in ["run_experiments.py", "profile_memory.py", "make_figures.py", "audit_data.py"]:
        subprocess.run([sys.executable, str(ROOT / "scripts" / script)], cwd=ROOT, check=True)
check = subprocess.run([sys.executable, str(ROOT / "scripts/verify_results.py")],
                       cwd=ROOT, check=True, capture_output=True, text=True)
print(check.stdout)
display(json.loads((ROOT / "results/environment.json").read_text()))

### 14. Conclusion et suite
Les méthodes classiques sont compactes et rapides, mais la couleur seule
confond des régions du décor et manque de contexte. Les améliorations dépendent
de la perturbation et ne justifient pas une supériorité universelle.

**Travail personnel à poursuivre :** expliquer un échec, formuler une nouvelle
hypothèse puis la tester sur une nouvelle validation ou un nouveau corpus.
Une comparaison avec un réseau léger serait une extension possible, avec
les mêmes partitions et mesures CPU/mémoire. Aucun résultat de réseau n'est
inventé dans ce projet.

**Références :** Tan et al. (2012), Jones et Rehg (2002), Kakumanu et al. (2007),
cours d'Eva Feillet et documentation OpenCV. Liens et attributions complets dans
`REFERENCES.md`. L'utilisation des données Pratheepan est non commerciale.